# 46500 part 2: introduction to Machine Learning - regression models
## 1 Introduction
This notebook provides an introduction to regression modelling using polynomials and simple ML methods.

We will go through a practical exercise, where the aim is characterizing the behaviour of the extreme loads at the root of a wind turbine blade and determining what is the dependence between these loads and environmental conditions (mean wind speed, turbulence, and wind shear). 
You are provided with a data set (ML_ExampleDataSet.xlsx) where 10-minute maxima for several wind turbine load channels are given as function of environmental conditions. 
Your main task is to:
-	Calibrate a regression model which maps the dependence between the loads and the environmental conditions. 
In addition, consider the following:
-	How would you evaluate the model performance?
-	What about trying different machine learning approaches – which one provides the best performance? 
-	For a particular choice of ML model, what hyperparameters provide the best performance, and how much data are necessary for achieving optimal performance?


Suggested procedure: <br>
1)	Load and parse input data; <br>
2)	Carry out explorative data analysis. Are there any visualisations which give a good overview of the data? Is any data filtering necessary?<br>
3)	Choose the type(s) of machine learning models you want to calibrate – what would an appropriate choice be, given the type of data you have, and given the required output?<br>
4)	Calibrate the surrogate model(s) of your choice<br>
5)	Make a residual analysis (plot residuals vs. variables) to check model adequacy, calculate error terms (e.g. r-square or RMS error) to evaluate the quality of the model fit and estimate model uncertainty <br>
6)	Explore changing hyperparameters and also running a convergence study – i.e., can we obtain a well-performing model with a fraction of the available data? <br>


## 2 Implementation

### 2.1 Data preparation

We will use a Pandas data frame which is a popular and efficient way of manipulating tabular data in Python.

In [1]:
import pandas as pd

In the cell below, enter code which loads the data from the sheet 'InputVariables' from the Excel file 'ML_ExampleDataSet.xlsx', and outputs a Pandas data frame called 'InputData'. Make sure the index of the 'InputData' frame is the 'Sample_No' column. Then you can display the first few rows of the dataframe using the DataFrame.head() command, where 'DataFrame' is replaced by the name of your dataframe object.

In [ ]:
InputData = pd.read_excel('ML_ExampleDataSet.xlsx','InputVariables')
InputData = InputData.set_index('Sample_No',drop = False) # Make the "Sample_No" column as index of the data
InputData # Show the first few rows of the data

In the cell below, enter code which loads the data from the sheet 'LoadResults' from the Excel file 'ML_ExampleDataSet.xlsx', and outputs a Pandas data frame called 'TargetData'. Make sure the index of the 'TargetData' frame is the 'PointNo' column.

In [ ]:
TargetData = pd.read_excel('ML_ExampleDataSet.xlsx','LoadResults')
TargetData.set_index('PointNo', drop = False, inplace = True) # Make the "PointNo" column as index of the data
TargetData # Show the first few rows of the data

Now we have two data frames, which contain respectively the input variables and the dependent variables for our machine learning problem. However, we are not sure whether these are appropriately synchronized, i.e., whether the indexes of the two data frames match. Pandas provides several means of comparing data columns and merging data frames. One way of doing it is with the df.merge() command where df is the name of the data frame. Here we will use an "inner" merging, which means that only data entries which exist in both data frames to be merged, are being retained. In this way the merging also acts as a filter to eliminate entries where some of the data may not be available. In supervised machine learning, useful information is provided by from data entry pairs where both the input variables and the dependent variable are valid. If one finds it more convenient to keep the input feature and dependent variable data frames separate, a similar synchronization can be done by element-wise comparison of the data frames, using the df.where() command. The merge approach is slightly faster, as it requires fewer comparisons in the background.

The merge of the two dataframes is done in the cell below. 


In [ ]:
AllInputData = InputData.where(InputData['Sample_No']==TargetData['PointNo'])
AllTargetData = TargetData.where(TargetData['PointNo']==InputData['Sample_No'])
AllInputData.drop(columns = 'Sample_No', inplace = True)
AllTargetData.drop(columns = 'PointNo', inplace = True)
nsamples = AllInputData['U'].count() # Find the total number of data points in the data frame
FeatureNames = AllInputData.columns.values
DependentVariableNames = AllTargetData.columns.values
print('Feature names: ', FeatureNames)
print('Dependent variable names: ', DependentVariableNames)
print(AllInputData.iloc[:,0].values)

In [ ]:
AllInputData

Let us visualize some of the features in the input data and the dependent variables. First, we'll make a scatter plot of the first 500 entries from the input variables. 

In [ ]:
import matplotlib.pyplot as plt
nplotpoints = 500
plt.rc('font', size=12) 
fig1,axs1 = plt.subplots(2,3,figsize = (16,10))
plt.setp(axs1,xlabel = "Mean wind speed [m/s]")
plt.setp(axs1[0,0], title = FeatureNames[0])
plt.setp(axs1[0,1], title = FeatureNames[1])
plt.setp(axs1[0,2], title = FeatureNames[2])
plt.setp(axs1[1,0], title = FeatureNames[3])
plt.setp(axs1[1,1], title = FeatureNames[4])
plt.setp(axs1[1,2], title = FeatureNames[5])
axs1[0,0].hist(AllInputData.iloc[:,0],50,color = 'C0')
axs1[0,1].plot(AllInputData.iloc[0:nplotpoints,0],AllInputData.iloc[0:nplotpoints,1],'o',color = 'C1')
axs1[0,2].plot(AllInputData.iloc[0:nplotpoints,0],AllInputData.iloc[0:nplotpoints,2],'o',color = 'C2')
axs1[1,0].plot(AllInputData.iloc[0:nplotpoints,0],AllInputData.iloc[0:nplotpoints,3],'o',color = 'C3')
axs1[1,1].plot(AllInputData.iloc[0:nplotpoints,0],AllInputData.iloc[0:nplotpoints,4],'o',color = 'C4')
axs1[1,2].plot(AllInputData.iloc[0:nplotpoints,0],AllInputData.iloc[0:nplotpoints,5],'o',color = 'C5')
plt.tight_layout()             
plt.show()

What is the relation between input variables and outputs? Since the wind speed is typically the most important variable, let us plot a few of the output variables as functions of the wind speed.

In [ ]:
fig2 = plt.figure(2, figsize = (18,9))

for i in range(DependentVariableNames.shape[0]):
    axi = fig2.add_subplot(2,4,i+1)
    plt.title(DependentVariableNames[i])
    plt.plot(AllInputData.U,AllTargetData.iloc[:,i],'.',markersize = 3)
    plt.xlabel('Mean wind speed [m/s]')
    #plt.ylabel(DependentVariableNames[i])
plt.tight_layout()
plt.show()

In [ ]:
fig3 = plt.figure(3, figsize = (18,9))

for i in range(DependentVariableNames.shape[0]):
    axi = fig3.add_subplot(2,4,i+1)
    plt.title(DependentVariableNames[i])
    plt.plot(AllInputData.SigmaU,AllTargetData.iloc[:,i],'.',markersize = 3)
    plt.xlabel('Wind standard deviation [m/s]')
    #plt.ylabel(DependentVariableNames[i])
plt.tight_layout()
plt.show()

### 2.2 Fitting a polynomial model
Let us first try to fit a simple polynomial model of the type $y = ax_1^n + bx_1^{n-1} + \ldots + zx_m^n + \ldots$.

A model of this type can be represented in matrix form as:

$\mathbf{y} = \mathbf{X}{\beta}$

where $\mathbf{y}$ is a vector with outputs, $\boldsymbol{\beta}$ is a vector with the polynomial coefficients, and $\mathbf{X}$ is the design matrix containing combinations of input variables raised to various powers. The size of the design matrix is $[m,n]$ where $m$ is the number of data points to be predicted, and $n$ is the number of polynomial coefficients. 
The polynomial coefficients, $\boldsymbol{\beta}$, can be determined by considering pairs of measured (or simulated) inputs and outputs. For a sample of input data factored in a design matrix $\mathbf{X}$, and a corresponding set of outputs $\hat{\mathbf{y}}$, under certain conditions the polynomial coefficients $\boldsymbol{\beta}$ can be determined by a closed-form expression:

$\boldsymbol{\beta} = \left( \mathbf{X}^T \mathbf{X} \right)^{-1} \mathbf{X}^T\hat{\mathbf{y}}$

Note that the inversion is normally an inefficient operation in computing, that's why typically other algebraic methods are applied to solve an equation of the type $\mathbf{A}\mathbf{x} = \mathbf{b}$. Instead of computing $\mathbf{x} = inv(\mathbf{A})*\mathbf{b}$, we can in python use the command *x = np.linalg.lstsq(A,b)*

Another hint: matrix multiplication with python can be easily done using the *np.dot()* command, for example computing $\mathbf{A}*\mathbf{B}$ could be done with *np.dot(A,B)*

In [ ]:
# Building a design matrix for a polynomial of 3rd order
def DesignMatrixO3(X):
    ndim = X.shape[1] 
    npoints = X.shape[0]
    m = int(((ndim-1)/2)*ndim)
    Xmatrix = np.zeros((npoints,3*ndim + 3*m + 1))
    columncount = 0
    Xmatrix[:,columncount] = np.ones(npoints)
    for i in range(ndim):
        columncount+=1
        Xmatrix[:,columncount] = X[:,i]

    for i in range(ndim -1):
        for j in range(i+1,ndim):
            columncount+= 1
            Xmatrix[:,columncount] = X[:,i]*X[:,j]

    for i in range(ndim):
        columncount+= 1
        Xmatrix[:,columncount] = X[:,i]**2

    for i in range(ndim-1):
        for j in range(i+1,ndim):
            columncount+= 1
            Xmatrix[:,columncount] = (X[:,i]**2)*X[:,j]

    for i in range(ndim-1):
        for j in range(i+1,ndim):
            columncount+= 1
            Xmatrix[:,columncount] = X[:,i]*(X[:,j]**2)

    for i in range(ndim):
        columncount+=1
        Xmatrix[:,columncount] = X[:,i]**3
    return Xmatrix


def PredictPolyO3(X,Alsq):
    Xmatrix = DesignMatrixO3(X)
    Y = np.dot(Xmatrix,Alsq)
    return Y

In [ ]:
AllInputData.values

In [ ]:
# MAKE A 3-RD ORDER POLYNOMIAL FIT TO THE DATA USING THE HELPER FUNCTIONS GIVEN ABOVE
import numpy as np
Y1 = AllTargetData['Tower_base_fore_aft_M_x']
X = AllInputData.values

# BEGIN CODE HERE
Xmatrix = 

XX = 
XY = 
Alsq = 
Alsq.shape

Ypred_O3 = 
# END CODE HERE

In [ ]:
# VISUALISE RESULTS FROM THE PREDICTION
plt.rc('font', size=14) 
fig2a,axs2a = plt.subplots(1,2,figsize = (16,8))
plt.setp(axs2a[0], title = 'Dependence vs. wind speed', xlabel = 'Mean wind speed [m/s]',ylabel = 'Tower base fore-aft moment $M_x$')
plt.setp(axs2a[1], title = 'Correlation (y-y) plot', xlabel = 'Input data',ylabel = 'Model predictions')
axs2a[0].plot(AllInputData['U'],AllTargetData['Tower_base_fore_aft_M_x'],'o',markersize = 4,color = 'C1')
axs2a[0].plot(AllInputData['U'],Ypred_O3,'*',markersize = 4,color = 'purple')
axs2a[0].legend(['Input data','Model predictions'])
axs2a[1].plot(AllTargetData['Tower_base_fore_aft_M_x'],Ypred_O3,'ok',markersize = 4)
axs2a[1].plot(np.array([np.min(AllTargetData['Tower_base_fore_aft_M_x']), np.max(AllTargetData['Tower_base_fore_aft_M_x'])]),\
             np.array([np.min(AllTargetData['Tower_base_fore_aft_M_x']), np.max(AllTargetData['Tower_base_fore_aft_M_x'])]),'-r')
axs2a[1].legend(['Point-to-point comparisons','1:1 relation'])
plt.tight_layout()             
plt.show()

#### Model performance assessment
Now we have trained a simple ML (regression) model. The plots are one way to assess its performance, but we normally want quantitative comparisons too. In our slides, we have listed a couple of popular measures of regression model performance - including R-squared, RMSE, and MAE. Let us compute these below (and also repeat the computation for any other model we train).

The formulas are:

$r^2 = 1 - \left(\frac{\sum_i{(y_i - g(x_i))^2}}{\sum_i{(y_i - \bar{y})^2}} \right)$

$RMSE = \sqrt{MSE} = \sqrt{\mathbb{E}\left( (y - g(x))^2\right)} = \sqrt{\frac{\sum_i{\left(y_i - g(x_i)\right)^2}}{N}}$

$MAE = \sum_{i=1}^N{\frac{|y_i - g(x_i)|}{N}}$

Interestingly, for well-performing models, the R-squared is approximately equal to the correlation coefficient between the target data and the model predictions. 

In [ ]:
Rsq_poly = 1 - np.sum( (Y1 - Ypred_O3)**2) / np.sum((Y1 - np.mean(Y1))**2)
Rsq_corr_poly = (np.corrcoef(Y1,Ypred_O3)[0,1])**2 # R-squared with the correlation coefficient
RMSE_poly = np.sqrt( np.mean((Y1 - Ypred_O3)**2))
print('R-square of polynomial model: ' + str(Rsq_poly))
print('R-square computed through correlations: ' + str(Rsq_corr_poly))
print('RMSE of polynomial model: ' + str(RMSE_poly))

### 2.3. Fitting a Random Forest model

Random forest (RF) is a popular Machine Learning method. Its popularity is due to the simplicity (very few hyperparameters) and good performance on simple modelling tasks. 
As discussed in the accompanying slides, Random Forest is essentially a combination of decision trees where the output of each tree is determined by "branching" conditions (logical "<" and ">"). The tree-based algorithm natively belongs to the family of discrete models (why? can you discuss?) - with some adaptations making it capable of approximating continuous functions. The discrete origin of the RF model may pose some challenges for using RF-based "surrogate models" for optimization. Could you think why this may be the case?

In [ ]:
import sklearn
RFregressor = sklearn.ensemble.RandomForestRegressor(n_estimators = 100)
RFregressor.fit(X,Y1)

In [ ]:
Yout = RFregressor.predict(X)
RFregressor.score(X,Y1)

In [ ]:
# VISUALISE RESULTS FROM THE PREDICTION
plt.rc('font', size=14) 
fig2b,axs2b = plt.subplots(1,2,figsize = (16,8))
plt.setp(axs2b[0], title = 'Dependence vs. wind speed', xlabel = 'Mean wind speed [m/s]',ylabel = 'Tower base fore-aft moment $M_x$')
plt.setp(axs2b[1], title = 'Correlation (y-y) plot', xlabel = 'Input data',ylabel = 'Model predictions')
axs2b[0].plot(AllInputData['U'],AllTargetData['Tower_base_fore_aft_M_x'],'o',markersize = 4,color = 'C1')
axs2b[0].plot(AllInputData['U'],Yout,'*',markersize = 4,color = 'purple')
axs2b[0].legend(['Input data','Model predictions'])
axs2b[1].plot(AllTargetData['Tower_base_fore_aft_M_x'],Yout,'ok',markersize = 4)
axs2b[1].plot(np.array([np.min(AllTargetData['Tower_base_fore_aft_M_x']), np.max(AllTargetData['Tower_base_fore_aft_M_x'])]),\
             np.array([np.min(AllTargetData['Tower_base_fore_aft_M_x']), np.max(AllTargetData['Tower_base_fore_aft_M_x'])]),'-r')
axs2b[1].legend(['Point-to-point comparisons','1:1 relation'])
plt.tight_layout()             
plt.show()

#### Question:
 Was the above the best practice approach of fitting and testing ML models? What else do we need to consider when testing the performance?

### 2.4 Fitting an Artificial Neural Network model

In [ ]:
import sklearn
import sklearn.neural_network

In the cell below, we will start using the scikit-learn toolbox which provides possibilities for fitting Artificial Neural Network models. Let us start with defining a regression model (*sklearn.neural_network.MLPRegressor*) - using a hyperbolic tangent activation function, and with two hidden layers with 12 perceptrons each. Then we can also play with other hyperparameters: batch size, initial learning rate, and other.

In [ ]:
# SKLEARN Neural Network MLP regressor model
# BEGIN CODE HERE
ANNmodel = 
# END CODE HERE
ANNmodel.get_params()

Let us try to quickly fit a model with activation = tanh, and hidden layer size 10 (fill in the spaces in the cell below). 

In [ ]:
# BEGIN CODE HERE
ANNmodel.set_params()

# END CODE HERE

In [ ]:
ANNmodel.score(AllInputData,Y1)

Does the model with tanh activation converge? Let us also try with a relu activation:

In [ ]:
# BEGIN CODE HERE
ANNmodel.set_params()

# END CODE HERE
print('Model r-square with full data set: ' + str(ANNmodel.score(AllInputData,AllTargetData['Tower_base_fore_aft_M_x'])))

In [ ]:
ANNmodel.score(AllInputData,Y1)

If one or more of the above fits did not work, what is the possible reason?
Now let us try to tweak our data a bit and make it better suitable for our ML model. In the cell below, try to normalize both the input and output data using the *sklearn.preprocessing.StandardScaler()* method:

In [ ]:

#X = (AllInputData - AllInputData.mean(axis=0))/AllInputData.std(axis=0)

Xscaler = sklearn.preprocessing.StandardScaler()
Yscaler = sklearn.preprocessing.StandardScaler()

# BEGIN CODE HERE
Xscaler = 
Yscaler = 

TrainTestRatio = 0.8
N = len(AllInputData)
Xtrain = 
Xtest = 

Ytrain = 
Ytest = 

# END CODE HERE

# UNCOMMENT TO REPLACE THE ABOVE CODE WITH A VERSION THAT USES ALL DATA FOR TRAINING
#Xtrain = Xscaler.transform(AllInputData.values)
#Ytrain = Yscaler.transform(AllTargetData['Tower_base_fore_aft_M_x'].values.reshape(-1,1))
#Xtest = Xtrain
#Ytest = Ytrain





In [ ]:
Xtest.shape

Now, let us make a fit using the transformed data:

In [ ]:
ANNmodel.set_params(learning_rate_init = 0.01, activation = 'relu',tol = 1e-6,n_iter_no_change = 10, hidden_layer_sizes = (12,12), validation_fraction = 0.1)
# BEGIN CODE HERE
ANNmodel.fit()
# END CODE HERE

In [ ]:
print( 'Train set r-square: ' + str(ANNmodel.score(Xtrain,Ytrain)))
print( 'Test set r-square: ' + str(ANNmodel.score(Xtest,Ytest)))

In [ ]:
# COMPUTE MODEL PREDICTIONS FOR ALL TEST DATA:
# BEGIN CODE HERE
Yout = Yscaler.inverse_transform(ANNmodel.predict(Xtrain).reshape(-1, 1))
Yout_test = Yscaler.inverse_transform(ANNmodel.predict(Xtest).reshape(-1, 1))
# END CODE HERE

In [ ]:
plt.hist(Yout,50)
plt.show()

Let us compare the model predictions with the target data:

In [ ]:
import numpy as np
plt.rc('font', size=14) 
fig3,axs3 = plt.subplots(1,2,figsize = (16,8))
plt.setp(axs3[0], title = 'Dependence vs. wind speed', xlabel = 'Mean wind speed [m/s]',ylabel = 'Tower base fore-aft moment $M_x$')
plt.setp(axs3[1], title = 'Correlation (y-y) plot', xlabel = 'Input data',ylabel = 'Model predictions')
axs3[0].plot(Xtest[:,0],Yscaler.inverse_transform(Ytest),'o',markersize = 4,color = 'C1')
axs3[0].plot(Xtest[:,0],Yout_test,'*',markersize = 4,color = 'purple')
axs3[0].legend(['Input data','Model predictions'])
axs3[1].plot(Yscaler.inverse_transform(Ytest),Yout_test,'ok',markersize = 4)
axs3[1].plot(np.array([np.min(AllTargetData['Tower_base_fore_aft_M_x']), np.max(AllTargetData['Tower_base_fore_aft_M_x'])]),\
             np.array([np.min(AllTargetData['Tower_base_fore_aft_M_x']), np.max(AllTargetData['Tower_base_fore_aft_M_x'])]),'-r')
axs3[1].legend(['Point-to-point comparisons','1:1 relation'])
plt.tight_layout()             
plt.show()

In [ ]:
plt.plot(Yout_test,Yscaler.inverse_transform(Ytest)/Yout_test, 'xk')